# DeFoG — Marginal Ablation on QM9 (Colab)

Self-contained notebook: builds a fresh `defog` conda environment every session,
then runs the two-way marginal ablation. No tarball caching, no setup-once notebook —
the whole pipeline is in this file.

### Prerequisites
- Runtime set to **GPU** (Runtime → Change runtime type → T4/A100).
- `spminer_motif_marginals.pt` is in `MyDrive/DeFoGColab/data/qm9/` (uploaded once via Drive web UI).
- `WANDB_API_KEY` saved as a Colab Secret (left sidebar → 🔑 Secrets).
- `GITHUB_REPO` cell below points to your fork.

### Expected runtime
- Environment build: ~8–12 min
- Ablation (2 × 100 epochs + test): ~3–5 h on T4, ~1.5–2 h on A100

## 1 · Environment setup

In [ ]:
# Mount Drive and load W&B API key from Colab Secrets.
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception as e:
    print(f'Warning: could not load WANDB_API_KEY ({e}).')
    print('Set manually if needed: os.environ["WANDB_API_KEY"] = "<your-key>"')

# MPLBACKEND must be Agg in Colab — Jupyter's default inline backend doesn't
# exist inside the conda env and breaks matplotlib's validator.
os.environ['MPLBACKEND'] = 'Agg'

print('Drive mounted.')

In [ ]:
# Verify GPU. The CUDA version determines which PyTorch wheel will be used
# (colab_create_env.sh detects this automatically).
!nvidia-smi

In [ ]:
%%bash
# Install Miniforge (conda-forge's installer — no Anaconda Terms-of-Service).
# Skipped on re-runs within the same session. ~30s on first install.
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed."
    /content/miniconda3/bin/conda --version
    exit 0
fi
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

In [ ]:
# Configure — set this to your fork's URL.
GITHUB_REPO = 'https://github.com/Maxlyu254/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')
print(f'       to : {REPO_DIR}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present — pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready."
git -C "$REPO_DIR" log --oneline -3

In [ ]:
%%bash
# Build the defog conda environment from scratch.
# ~8-12 min: ~2-3 min mamba solve, ~3-5 min download, ~2-3 min pip layer.
# All known dependency-conflict workarounds are baked into colab_create_env.sh.
set -e
bash /content/DeFoGPlus/shell/colab_create_env.sh 2>&1

In [ ]:
%%bash -s "$REPO_DIR"
# Wire Drive paths via symlinks so checkpoints persist across sessions.
#   src/checkpoints/ -> Drive/DeFoGColab/checkpoints/
#   data/            -> Drive/DeFoGColab/data/
REPO_DIR="$1"
set -e

DRIVE_BASE='/content/drive/MyDrive/DeFoGColab'
mkdir -p "$DRIVE_BASE/checkpoints" "$DRIVE_BASE/data/qm9"

CKPT_LINK="$REPO_DIR/src/checkpoints"
[ -L "$CKPT_LINK" ] && rm "$CKPT_LINK"
ln -s "$DRIVE_BASE/checkpoints" "$CKPT_LINK"
echo "Symlink: $CKPT_LINK -> $DRIVE_BASE/checkpoints"

DATA_LINK="$REPO_DIR/data"
[ -L "$DATA_LINK" ] || [ -d "$DATA_LINK" ] && rm -rf "$DATA_LINK"
ln -s "$DRIVE_BASE/data" "$DATA_LINK"
echo "Symlink: $DATA_LINK -> $DRIVE_BASE/data"

echo "Symlinks ready."

In [ ]:
# Define BASE_ENV once — every subprocess in this notebook inherits these.
#
# LD_PRELOAD: conda's libgomp must load before torch's bundled libgomp.so.1
#   (older), otherwise graph_tool fails with `GOMP_5.0 not found`.
# MPLBACKEND: Colab's Jupyter inherits an inline backend that doesn't exist
#   in the conda env; Agg overrides it.
# PYTHONPATH: lets `from src import utils` resolve when running main.py from
#   inside src/ (sys.path[0] would otherwise point to src/, not the repo root).
import subprocess, os

PYTHON     = '/content/miniconda3/envs/defog/bin/python'
DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'
LIBGOMP    = '/content/miniconda3/envs/defog/lib/libgomp.so.1'

BASE_ENV = {
    **os.environ,
    'MPLBACKEND': 'Agg',
    'LD_PRELOAD': LIBGOMP,
    'PYTHONPATH': REPO_DIR,
}

# Verify imports work end-to-end with the full BASE_ENV.
check = '''
import torch, graph_tool, rdkit, torch_geometric, pytorch_lightning, hydra, wandb, imageio
print("graph_tool       :", graph_tool.__version__)
print("torch            :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
print("torch_geometric  :", torch_geometric.__version__)
print("pytorch_lightning:", pytorch_lightning.__version__)
print("imageio          :", imageio.__version__)
'''
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=BASE_ENV)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Import check failed.')

# W&B auth
r2 = subprocess.run([PYTHON, '-m', 'wandb', 'whoami'],
                    capture_output=True, text=True, env=BASE_ENV)
print('W&B identity:', r2.stdout.strip() or '(not logged in)')

# Precomputed .pt file
pt_file = f'{REPO_DIR}/data/qm9/spminer_motif_marginals.pt'
print(f'  {"✓" if os.path.isfile(pt_file) else "✗ MISSING"}  {pt_file}')

# Symlinks
for link, target in [
    (f'{REPO_DIR}/src/checkpoints', f'{DRIVE_BASE}/checkpoints'),
    (f'{REPO_DIR}/data',            f'{DRIVE_BASE}/data'),
]:
    ok = os.path.islink(link) and os.readlink(link) == target
    print(f'  {"✓" if ok else "✗"} symlink  {link}')

print('\nEnvironment ready.')

## 2 · Marginal Ablation

Two training + testing runs back-to-back, isolating the effect of the initial noise distribution:

| Run | Transition | Marginals source |
|-----|-----------|------------------|
| `qm9_ablation_marginal` | `marginal` | Dataset type-frequency statistics |
| `qm9_ablation_spminer`  | `loaded_marginal` | SPMiner-mined motif marginals |

Each run: 100 epochs training → test with last checkpoint → results to W&B.  
Checkpoints saved directly to Drive via the symlink wired above.

In [ ]:
# Helpers and shared Hydra overrides.
import glob

SRC      = f'{REPO_DIR}/src'
DATA_QM9 = f'{REPO_DIR}/data/qm9'
RUN_ENV  = BASE_ENV  # reuse the env defined above

# check_val_every_n_epochs=20  -> validate at epochs 20, 40, 60, 80, 100
# sample_every_val=1           -> checkpoint at every validation
# samples_to_generate=128      -> small mid-training sample (keeps val fast)
# final_model_samples_to_generate=512 -> test budget
COMMON = [
    '+experiment=qm9_no_h',
    'hydra.job.chdir=False',
    'train.n_epochs=100',
    'general.check_val_every_n_epochs=20',
    'general.sample_every_val=1',
    'general.samples_to_generate=128',
    'general.final_model_samples_to_generate=512',
]


def run_live(cmd, cwd=SRC):
    """Run cmd, streaming stdout+stderr line by line. Raise on non-zero exit."""
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=cwd, env=RUN_ENV,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Process exited with code {proc.returncode}')


def train_and_test(name, extra):
    """Train 100 epochs then test with the last checkpoint.

    Args:
        name:  W&B run name and checkpoint subdirectory.
        extra: Run-specific Hydra overrides (e.g. transition, marginals path).
    """
    base_args = COMMON + [f'general.name={name}'] + extra

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TRAIN  →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')
    run_live([PYTHON, 'main.py'] + base_args)

    ckpt_dir = f'{SRC}/checkpoints/{name}'
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'), key=os.path.getmtime)
    if not ckpts:
        raise RuntimeError(f'No checkpoint found in {ckpt_dir}.')
    ckpt = ckpts[-1]
    print(f'\nLast checkpoint: {ckpt}')

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TEST   →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')
    run_live([PYTHON, 'main.py'] + base_args + [f'general.test_only={ckpt}'])

    print(f'\n✓  {name} complete.  Checkpoint: {ckpt}')
    return ckpt


print('Helpers loaded.')
print(f'SRC      : {SRC}')
print(f'DATA_QM9 : {DATA_QM9}')

In [ ]:
# Pre-flight checks.
import torch

errors = []

if torch.cuda.is_available():
    print(f'✓  GPU: {torch.cuda.get_device_name(0)}')
else:
    errors.append('No CUDA GPU detected — training will be extremely slow.')

spminer_pt = f'{DATA_QM9}/spminer_motif_marginals.pt'
if os.path.isfile(spminer_pt):
    m = torch.load(spminer_pt, map_location='cpu', weights_only=True)
    print(f'✓  SPMiner marginals: X={list(m["X"].shape)}  E={list(m["E"].shape)}')
else:
    errors.append(f'Missing: {spminer_pt}')

if os.environ.get('WANDB_API_KEY', ''):
    print('✓  WANDB_API_KEY is set.')
else:
    print('⚠  WANDB_API_KEY not set — runs will not be logged to W&B.')

if errors:
    for e in errors:
        print(f'✗  {e}')
    raise RuntimeError('Pre-flight failed. Fix the issues above before continuing.')
print('\nAll checks passed — ready to train.')

In [ ]:
# Run 1 — unedited marginal distribution.
# Initial noise z_T sampled from the dataset's empirical type-frequency
# marginals (no motif editing). Standard DeFoG marginal transition.
ckpt_marginal = train_and_test(
    name  = 'qm9_ablation_marginal',
    extra = ['model.transition=marginal'],
)

In [ ]:
# Run 2 — SPMiner motif-edited marginal distribution.
# Initial noise z_T sampled from marginals estimated from QM9 graphs after
# injecting the top-1 SPMiner-mined motif (a 3-node chain). Only the rate
# matrix changes vs Run 1 — model architecture and graph size are identical.
SPMINER_PT = f'{DATA_QM9}/spminer_motif_marginals.pt'

ckpt_spminer = train_and_test(
    name  = 'qm9_ablation_spminer',
    extra = [
        'model.transition=loaded_marginal',
        f'model.motif_marginals_path={SPMINER_PT}',
    ],
)

In [ ]:
# Summary.
print('=' * 60)
print('  Ablation complete')
print('=' * 60)
print()
rows = [
    ('qm9_ablation_marginal', 'marginal',        ckpt_marginal),
    ('qm9_ablation_spminer',  'loaded_marginal', ckpt_spminer),
]
for run_name, transition, ckpt in rows:
    print(f'Run        : {run_name}')
    print(f'Transition : {transition}')
    print(f'Checkpoint : {ckpt}')
    print()

print('Results logged to W&B under the run names above.')
print(f'Checkpoints persisted at: {DRIVE_BASE}/checkpoints/')